# Практическая работа № 3. Сборка пайплайна предобработки

**Выполнил:** Алмасов Арсен

**Цель:** собрать пайплайн обработки данных без утечки информации из тестовой выборки.

## 1. Импорт библиотек и загрузка данных

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("customers_dirty.csv")
print("Размер данных:", df.shape)
display(df.head())
print("\\nПропуски по столбцам:")
print(df.isnull().sum())

Размер данных: (30, 6)


,customer_id,age,monthly_income,city,visits_per_month,satisfaction_score
0,1,19.0,18000.0,Бишкек,5,8.0
1,2,22.0,25000.0,Ош,8,9.0
2,3,25.0,32000.0,Бишкек,12,7.0
3,4,31.0,45000.0,Каракол,6,8.0
4,5,28.0,38000.0,Ош,10,9.0


\nПропуски по столбцам:
customer_id           0
age                   1
monthly_income        1
city                  0
visits_per_month      0
satisfaction_score    1
dtype: int64


## 2. Разделение на train и test

In [2]:
# Удаляем идентификатор: он не нужен для обучения преобразований
X = df.drop(columns=["customer_id"])

X_train, X_test = train_test_split(
    X, test_size=0.2, random_state=42
)

print("Размер train:", X_train.shape)
print("Размер test:", X_test.shape)

Размер train: (24, 5)
Размер test: (6, 5)


## 3. Настройка обработки признаков

In [3]:
numeric_features = [
    "age", "monthly_income", "visits_per_month", "satisfaction_score"
]
categorical_features = ["city"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Пайплайн настроен.")

Пайплайн настроен.


## 4. Обучение на train и преобразование test

In [4]:
# fit_transform обучает преобразования только на train
X_train_processed = preprocessor.fit_transform(X_train)

# test только преобразуется уже обученным пайплайном
X_test_processed = preprocessor.transform(X_test)

print("Размер обработанного train:", X_train_processed.shape)
print("Размер обработанного test:", X_test_processed.shape)
print("Одинаковое число признаков:",
      X_train_processed.shape[1] == X_test_processed.shape[1])

Размер обработанного train: (24, 8)
Размер обработанного test: (6, 8)
Одинаковое число признаков: True


## 5. Выводы

- Данные разделены на train и test в пропорции 80/20.
- Числовые пропуски заполняются медианой.
- Категориальные пропуски заполняются наиболее частым городом.
- Город кодируется с помощью OneHotEncoder.
- Числовые признаки масштабируются StandardScaler.
- Все преобразования объединены в Pipeline и ColumnTransformer.
- Преобразования обучаются только на train, поэтому статистики тестовой выборки не попадают в обучение.
- Размерности обработанных train и test проверены.